# **Instalação do Unsloth e dependências**

In [6]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes datasets huggingface_hub

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-fnpiegm4/unsloth_24e395f9aa0f4b45b22fda151ba79a93
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-fnpiegm4/unsloth_24e395f9aa0f4b45b22fda151ba79a93
  Resolved https://github.com/unslothai/unsloth.git to commit 091294901198f3f2c05020b8464e8d34020565b9
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 83.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 63.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 134.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 945.8/945.8 kB 48.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 85.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 130.0 MB/s  0:00:00
  Created wheel for unsl

# **Carregar o Modelo Base e o Tokenizer**

In [7]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None # Deteção automática (Float16 para GPU T4)
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-3B-Instruct-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W1111 18:11:33.109000 1242 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1111 18:11:33.162000 1242 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.11.4: Fast Qwen2 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.14.1+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.8.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

#**Adicionar os Adaptadores LoRA (QLoRA)**

In [8]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0, # Otimizado para 0 no Unsloth
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

Unsloth 2026.11.4 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


Unsloth: compiling each transformer block as one region (Qwen2DecoderLayer).


# **Carregamento do Dataset diretamente do Hugging Face**

In [9]:
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

# Aplica o padrão nativo de chat do Qwen2.5 (ChatML)
tokenizer = get_chat_template(
    tokenizer,
    chat_template="qwen-2.5",
)

def formatar_conversas(amostra):
    conversa = amostra["messages"]
    texto_formatado = tokenizer.apply_chat_template(conversa, tokenize=False, add_generation_prompt=False)
    return {"text": texto_formatado}

# Carrega o seu dataset publicado no Hub
dataset = load_dataset("fernandodlts/brazilian-delivery-absa-dataset", data_files="train.jsonl", split="train")
dataset = dataset.map(formatar_conversas, batched=False)

print(f"Total de exemplos de treino prontos: {len(dataset)}")
print("\nExemplo da primeira linha processada:")
print(dataset[0]["text"][:350] + "...")

train.jsonl:   0%|          | 0.00/278k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/240 [00:00<?, ? examples/s]

Total de exemplos de treino prontos: 240

Exemplo da primeira linha processada:
<|im_start|>system
Você é um especialista em análise de sentimento e extração de dados operacionais do setor de delivery (iFood).
Sua tarefa é analisar avaliações de clientes e extrair um objeto JSON estrito com os seguintes aspectos:
- comida: 'positivo', 'negativo', 'neutro' ou 'nao_mencionado'
- entrega: 'positivo', 'negativo', 'neutro' ou 'nao_...


# **Configuração e Execução do Treino**

In [10]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False, # Não agrupa textos curtos para não corromper o formato JSON
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=5,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="outputs",
        report_to="none",
    ),
)

print("A iniciar o ajuste fino supervisionado...")
trainer_stats = trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/240 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
A iniciar o ajuste fino supervisionado...


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 240 | Num Epochs = 3 | Total steps = 90
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: packed / padding-free rows run each transformer block uncompiled; only unpacked batches use the compiled blocks.
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
5,2.008016
10,1.369929
15,0.850503
20,0.586768
25,0.541959
30,0.517138
35,0.478175
40,0.471778
45,0.453112
50,0.403313


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-90/tokenizer_config.json.


# **Teste de inferência**

In [11]:
from unsloth import FastLanguageModel

# Ativa o modo de inferência acelerada nativo do Unsloth
FastLanguageModel.for_inference(model)

# Avaliação desafiante: gírias, comida boa, erro de separação, atraso crítico e risco de churn
review_teste = "o burger tava surreal de bom quentinho e suculento mas pow atrasou 1h e meia e esqueceram minha coca lata.. pedi estorno no suporte nunca mais peco nessa loja"

mensagens = [
    {
        "role": "system",
        "content": (
            "Você é um especialista em análise de sentimento e extração de dados operacionais do setor de delivery (iFood).\n"
            "Sua tarefa é analisar avaliações de clientes e extrair um objeto JSON estrito com os seguintes aspectos:\n"
            "- comida: 'positivo', 'negativo', 'neutro' ou 'nao_mencionado'\n"
            "- entrega: 'positivo', 'negativo', 'neutro' ou 'nao_mencionado'\n"
            "- embalagem: 'positivo', 'negativo', 'neutro' ou 'nao_mencionado'\n"
            "- custo_beneficio: 'positivo', 'negativo', 'neutro' ou 'nao_mencionado'\n"
            "- risco_churn: true ou false\n"
            "- resumo_critica: frase curta com o problema principal ou null se não houver problemas.\n\n"
            "Responda APENAS com o JSON válido, sem introduções ou explicações."
        )
    },
    {"role": "user", "content": review_teste}
]

entradas = tokenizer.apply_chat_template(
    mensagens,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
).to("cuda")

saida = model.generate(
    input_ids=entradas,
    max_new_tokens=256,
    temperature=0.1,  # Temperatura baixa para saída determinística e estruturada
    use_cache=True
)

texto_gerado = tokenizer.batch_decode(saida[:, entradas.shape[1]:], skip_special_tokens=True)[0]
print("--- RESPOSTA DO MODELO AJUSTADO ---")
print(texto_gerado)

Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--- RESPOSTA DO MODELO AJUSTADO ---
{"comida": "positivo", "entrega": "negativo", "embalagem": "nao_mencionado", "custo_beneficio": "nao_mencionado", "risco_churn": true, "resumo_critica": "Atraso de uma hora e meia e esquecimento da lata de refrigerante"}


# **Publicação dos Pesos LoRA no Hugging Face Hub**

In [12]:
from huggingface_hub import login

# Executa o login na sessão do Colab (cole seu token Write quando solicitado)
login()

repo_modelo = "fernandodlts/qwen2.5-3b-delivery-absa-lora"

print(f"Enviando adaptadores LoRA para: {repo_modelo}...")

# Salva apenas os adaptadores treinados (~100 MB) e a configuração do tokenizer
model.push_to_hub_merged(
    repo_modelo,
    tokenizer,
    save_method="lora",
    token=True
)

print(f"\nModelo salvo com sucesso no Hugging Face Hub!")
print(f"Acesse em: https://huggingface.co/{repo_modelo}")

Enviando adaptadores LoRA para: fernandodlts/qwen2.5-3b-delivery-absa-lora...
Unsloth: Saving LoRA adapters. Please wait...


README.md:   0%|          | 0.00/575 [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   2%|1         | 2.24MB /  120MB            

Validating                    :           |   0%            

... (2 outputs left out)

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpr11lm2k2/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

Validating                    :           |   0%            

Saved lora model to https://huggingface.co/fernandodlts/qwen2.5-3b-delivery-absa-lora

Modelo salvo com sucesso no Hugging Face Hub!
Acesse em: https://huggingface.co/fernandodlts/qwen2.5-3b-delivery-absa-lora


# **Avaliação Quantitativa**

In [13]:
import json
from datasets import load_dataset
from tqdm import tqdm
from collections import defaultdict

# 1. Carregar o conjunto de teste diretamente do Hugging Face Hub
print("A carregar o conjunto de teste...")
dataset_teste = load_dataset(
    "fernandodlts/brazilian-delivery-absa-dataset",
    data_files="test.jsonl",
    split="train" # O ficheiro individual é mapeado como partição de leitura
)

print(f"Total de amostras de teste para avaliar: {len(dataset_teste)}")

# 2. Configurar inferência e recolha de resultados
FastLanguageModel.for_inference(model)

predicoes = []
respostas_reais = []
falhas_json = 0

for item in tqdm(dataset_teste, desc="A avaliar amostras"):
    mensagens = item["messages"]

    # Extrai o ground truth (a resposta real da anotação)
    conteudo_esperado = json.loads(mensagens[2]["content"])
    respostas_reais.append(conteudo_esperado)

    # Prepara o prompt (apenas system + user)
    conversa_prompt = mensagens[:2]
    entradas = tokenizer.apply_chat_template(
        conversa_prompt,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda")

    # Gera a resposta do modelo
    saida = model.generate(
        input_ids=entradas,
        max_new_tokens=256,
        temperature=0.01, # Mínima aleatoriedade para teste determinístico
        use_cache=True
    )

    texto_resposta = tokenizer.batch_decode(saida[:, entradas.shape[1]:], skip_special_tokens=True)[0].strip()

    # Validação estrutural do JSON
    try:
        dados_previstos = json.loads(texto_resposta)
        predicoes.append(dados_previstos)
    except Exception:
        falhas_json += 1
        # Fallback neutro em caso de JSON malformado para cálculo de penalização
        predicoes.append({
            "comida": "nao_mencionado",
            "entrega": "nao_mencionado",
            "embalagem": "nao_mencionado",
            "custo_beneficio": "nao_mencionado",
            "risco_churn": False,
            "resumo_critica": None
        })

# 3. Cálculo de Acurácia por Campo
total_testes = len(respostas_reais)
aspetos = ["comida", "entrega", "embalagem", "custo_beneficio", "risco_churn"]
acertos = defaultdict(int)

for real, prev in zip(respostas_reais, predicoes):
    for campo in aspetos:
        if real.get(campo) == prev.get(campo):
            acertos[campo] += 1

taxa_json_valido = ((total_testes - falhas_json) / total_testes) * 100

print("\n" + "="*50)
print("       RELATÓRIO TÉCNICO DE AVALIAÇÃO (TEST SPLIT)")
print("="*50)
print(f"Amostras avaliadas:           {total_testes}")
print(f"Aderência a JSON estrito:     {taxa_json_valido:.2f}% ({total_testes - falhas_json}/{total_testes})")
print("-"*50)
print("Acurácia por Dimensão Operacional:")
for campo in aspetos:
    taxa_acerto = (acertos[campo] / total_testes) * 100
    print(f" - {campo.ljust(16)}: {taxa_acerto:.2f}% ({acertos[campo]}/{total_testes})")
print("="*50)

A carregar o conjunto de teste...


test.jsonl:   0%|          | 0.00/70.4k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Total de amostras de teste para avaliar: 60


A avaliar amostras: 100%|██████████| 60/60 [01:43<00:00,  1.72s/it]


       RELATÓRIO TÉCNICO DE AVALIAÇÃO (TEST SPLIT)
Amostras avaliadas:           60
Aderência a JSON estrito:     100.00% (60/60)
--------------------------------------------------
Acurácia por Dimensão Operacional:
 - comida          : 85.00% (51/60)
 - entrega         : 100.00% (60/60)
 - embalagem       : 90.00% (54/60)
 - custo_beneficio : 81.67% (49/60)
 - risco_churn     : 98.33% (59/60)
